In [1]:
# 配置：不会调用模型。
import sys
from pathlib import Path
PROJECT = Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")
for location in (PROJECT, PROJECT.parent / "demiflow"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))
import json, os
from pathlib import Path
from project import resolve_root
from curation.image_facts.image_facts_pipeline import config, run_pipeline
ROOT = resolve_root()
CONFIG = config(**json.loads((ROOT / "demiwtg/curation/image_facts/configs/pilot10.json").read_text()))
RUN_PIPELINE = False
EXECUTE_MODELS = ["qwen4b"]  # 每个模型可用独立 CLI 并行提交，同模型不能重叠。
FINALIZE_ONLY = False
os.environ.setdefault("IMAGE_FACTS_LOCAL_KEY", "local-no-auth")
os.environ.setdefault("MODELHUB_API_KEY", "local-no-auth")
CONFIG

{'run': 'pilot10_64_three_models_v2_20261004',
 'source': {'uri': 'demiwtg/curation/t2i_positive_pairs/datasets/inputs_v4__pilot10_target1024_20261004.lance',
  'version': 3},
 'models': ['gpt61', 'luna', 'qwen4b'],
 'reference_model': 'gpt61',
 'prompt_config': '/yzp/zhaozy/yangzepeng/0905/demiwtg/curation/image_facts/prompts/tasks.yaml',
 'batch_sizes': {'gpt61': 8, 'luna': 8, 'qwen4b': 1},
 'concurrency': {'gpt61': 2, 'luna': 2, 'qwen4b': 8},
 'prompt_options': {'gpt61': {'stream': True,
   'timeout_s': 900,
   'gateway': 'litellm',
   'request_options': {'max_completion_tokens': 32768,
    'reasoning_effort': 'xhigh'}},
  'luna': {'stream': True,
   'timeout_s': 900,
   'gateway': 'litellm',
   'request_options': {'max_completion_tokens': 32768,
    'reasoning_effort': 'xhigh'}},
  'qwen4b': {'stream': True,
   'timeout_s': 300,
   'request_options': {'max_tokens': 8192,
    'temperature': 0,
    'chat_template_kwargs': {'enable_thinking': False}}}},
 'local_service': {'python': '/

In [2]:
# 唯一生产入口：设 RUN_PIPELINE=True 才调用模型并提交表。
if RUN_PIPELINE:
    RESULT = run_pipeline(CONFIG, execute_models=EXECUTE_MODELS, finalize_only=FINALIZE_ONLY)
    display(RESULT)
else:
    print("仅查看已有结果，未发起调用。")

仅查看已有结果，未发起调用。


In [ ]:
# 独立只读：每个具体标签独立统计；COMMON_ONLY=False 可查看各模型与参照的成对有效子集。
import sys
from pathlib import Path
PROJECT = Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")
for location in (PROJECT, PROJECT.parent / "demiflow"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))
import json
import lance
import pandas as pd
from project import resolve_root
from IPython.display import display
COMMON_ONLY = True
SUMMARY_PATH = resolve_root() / "demiwtg/curation/image_facts/datasets/summary__pilot10_64_three_models_v2_20261004.lance"
if not SUMMARY_PATH.exists():
    print("本轮摘要尚未提交；请查看运行进度，未提交不代表成功。")
else:
    SUMMARY_TABLE = lance.dataset(str(SUMMARY_PATH))
    SUMMARY = SUMMARY_TABLE.to_table().to_pylist()[0]
    OUTPUTS = json.loads(SUMMARY["outputs_json"])
    print("摘要版本", SUMMARY_TABLE.version, "图片数", SUMMARY["image_count"], "全部有效", SUMMARY["complete"])
    MODEL_METRICS = pd.DataFrame(json.loads(SUMMARY["models_json"]))
    MODEL_METRICS["first_model_seconds"] = MODEL_METRICS.apply(lambda row: None if row.get("wall_seconds_is_replay") else row["wall_seconds"], axis=1)
    display(MODEL_METRICS[["model_key", "images", "ok", "requests", "input_tokens", "output_tokens", "first_model_seconds"]])
    print("Qwen时间包含模型加载；GPT首次阶段登记为缓存重放，留空，不将重放当作推理时间。首轮进度观测约14分20秒，见README。token按原生请求日志去重统计，包含失败调用已返回的usage。")
    VALID_SETS = []
    for model in json.loads(SUMMARY["config_json"])["models"]:
        labels = lance.dataset(**OUTPUTS["labels_"+model]).to_table(columns=["sha256", "status"]).to_pylist()
        VALID_SETS.append({row["sha256"] for row in labels if row["status"] == "ok"})
    COMMON_SHA = set.intersection(*VALID_SETS)
    COMPARISONS = lance.dataset(**OUTPUTS["comparisons"]).to_table().to_pandas()
    if "tag" not in COMPARISONS.columns:
        print("此摘要为旧的整组比较。请通过正式入口 --finalize-only 整理为逐标签结果；不需重跑模型。")
    else:
        SELECTED = COMPARISONS[COMPARISONS.sha256.isin(COMMON_SHA)] if COMMON_ONLY else COMPARISONS
        VALID = SELECTED[SELECTED.status == "comparable"]
        print("当前范围：", "三模型共同有效" if COMMON_ONLY else "各模型与GPT成对有效", "；共同有效图数", len(COMMON_SHA), "/", SUMMARY["image_count"])
        print("下表的未标出只描述模型输出，不证明图中事实不存在。GPT为参照，不是人工真值。正例召回=共同标出/GPT标出；标出精确率=共同标出/本模型标出；分母为0显示无样本。")
        FIELD_NAMES = {"composition":"主体组合形式", "framing":"取景", "view_tags":"视角", "relation_types":"对象关系类型", "observability_issue_types":"可见性问题类型", "text_presence":"文字是否可见", "text_kind":"文字类型", "watermark_presence":"覆盖水印是否可见"}
        AGREEMENT_ROWS = []
        for (model, field, tag), group in VALID.groupby(["model_key", "field", "tag"], sort=False):
            marked = group.model_selected.astype(bool)
            reference = group.reference_selected.astype(bool)
            together = int((marked & reference).sum())
            AGREEMENT_ROWS.append({"model":model, "field":field, "tag":tag, "n":len(group),
                "reference_count":int(reference.sum()), "model_count":int(marked.sum()), "both":together,
                "reference_only":int((reference & ~marked).sum()), "model_only":int((marked & ~reference).sum()),
                "neither":int((~marked & ~reference).sum()),
                "recall":together / reference.sum() if reference.sum() else None,
                "precision":together / marked.sum() if marked.sum() else None})
        TAG_METRICS = pd.DataFrame(AGREEMENT_ROWS)
        for field, title in FIELD_NAMES.items():
            print(title, "（每个具体值单列）")
            table = TAG_METRICS[TAG_METRICS.field == field].drop(columns="field").copy() if len(TAG_METRICS) else pd.DataFrame()
            if len(table):
                for name in ("recall", "precision"):
                    table[name] = table[name].map(lambda value: "无样本" if pd.isna(value) else f"{value:.1%}")
                table = table.rename(columns={"model":"模型", "tag":"具体标签", "n":"有效图数", "reference_count":"GPT标出", "model_count":"本模型标出", "both":"共同标出", "reference_only":"仅GPT标出", "model_only":"仅本模型标出", "neither":"两者均未标出", "recall":"正例召回率", "precision":"标出精确率"})
            with pd.option_context("display.max_rows", None, "display.max_columns", None):
                display(table)
        print("关系仅统计某种类型有没有被标出；关系双方、方向和依据须逐图查看。对象自由名称、caption、文字内容、水印位置和遮挡说明不作字符串准确率。")
        print("水印指后加的署名、版权、图库商标等覆盖层；实物商标、招牌、棋盘文字不自动算水印。上表仅比较有/未观察到/无法确认，不比较位置或遮挡程度。已有结果未单独收集水印子类型、面积比例或遮挡等级。")


In [ ]:
# 独立只读：逐项查看具体标签，不折叠为整个 annotation JSON。
# 改 IMAGE_INDEX 查看另一张；CALL_MODEL 选 gpt61/luna/qwen4b 查看整批真实请求。
import sys
from pathlib import Path
PROJECT = Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")
for location in (PROJECT, PROJECT.parent / "demiflow"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))
import json, base64
import lance
import pandas as pd
from project import resolve_root
from demiflow.objects import ObjectRef
from demiflow.operator_llm.call_ref import read_call
from IPython.display import display, Image, Markdown, JSON
IMAGE_INDEX = 32  # 骑手与马：Qwen对象列表和组合形式存在不一致。
CALL_MODEL = None
SUMMARY_PATH = resolve_root() / "demiwtg/curation/image_facts/datasets/summary__pilot10_64_three_models_v2_20261004.lance"
if not SUMMARY_PATH.exists():
    print("本轮摘要尚未提交；请查看运行进度，未提交不代表成功。")
else:
    SUMMARY = lance.dataset(str(SUMMARY_PATH)).to_table().to_pylist()[0]
    OUTPUTS = json.loads(SUMMARY["outputs_json"])
    SCOPE = lance.dataset(**OUTPUTS["scope"])
    if SCOPE.count_rows() == 0:
        print("本轮无图片")
    elif "label_fields_gpt61" not in OUTPUTS:
        print("此摘要未拆分明细；请先通过正式入口 --finalize-only 整理已有结果。")
    else:
        ITEM = SCOPE.take([IMAGE_INDEX]).to_pylist()[0]
        print("图片", IMAGE_INDEX+1, "/", SCOPE.count_rows(), ITEM["sha256"])
        display(pd.DataFrame(json.loads(ITEM["bindings_json"])))
        display(Image(data=ObjectRef(ITEM["image_uri"], ITEM["sha256"]).read(max_bytes=64*1024**2), width=700))
        LABELS = {}
        for model in json.loads(SUMMARY["config_json"])["models"]:
            found = lance.dataset(**OUTPUTS["label_fields_"+model]).to_table(filter="sha256 = '"+ITEM["sha256"]+"'").to_pylist()
            LABELS[model] = found[0] if found else {"status":"missing", "error":"没有已提交标注"}
        FIELDS = {"status":"技术状态", "error":"失败说明", "width":"原图宽", "height":"原图高", "composition":"主体组合形式", "framing":"取景", "text_presence":"文字：是否可见", "text_kind":"文字：类型", "text_content":"文字：可读内容", "text_location":"文字：位置", "text_detail":"文字：观察限制", "watermark_presence":"覆盖水印：是否可见", "watermark_location":"覆盖水印：位置", "watermark_detail":"覆盖水印：覆盖与遮挡说明", "caption":"详细caption"}
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None, "display.max_columns", None):
            display(pd.DataFrame({model:{title:label.get(field) for field,title in FIELDS.items()} for model,label in LABELS.items()}))
        print("视角：每个标签独立显示；未标出不等于视觉事实不存在。")
        VIEWS = lance.dataset(**OUTPUTS["comparisons"]).to_table(columns=["tag"], filter="field = 'view_tags'").column("tag").unique().to_pylist()
        display(pd.DataFrame({model:{tag:("不可比较" if label["status"] != "ok" else "标出" if tag in label["view_tags"] else "未标出") for tag in VIEWS} for model,label in LABELS.items()}))
        for model, label in LABELS.items():
            print(model)
            if label["status"] != "ok":
                print("标注失败或缺失，不将空值解释为没有对象、关系或可见性问题。", label.get("error"))
            else:
                print("主要对象：每个对象一行")
                OBJECT_NAMES = {item["object_id"]:item["name"] for item in label["objects"]}
                with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
                    display(pd.DataFrame(label["objects"], columns=["object_id","name","location","visible_features","count"]).rename(columns={"object_id":"图内ID","name":"名称","location":"位置","visible_features":"可见特征","count":"数量（未知为空）"}))
                    print("对象关系：每条关系一行（未标出关系）" if not label["relations"] else "对象关系：每条关系一行")
                    relation_rows = [{"主体":OBJECT_NAMES.get(item["subject_id"]), "主体ID":item["subject_id"], "关系":item["type"], "客体":OBJECT_NAMES.get(item["object_id"]), "客体ID":item["object_id"], "可见依据":item["evidence"]} for item in label["relations"]]
                    display(pd.DataFrame(relation_rows, columns=["主体","主体ID","关系","客体","客体ID","可见依据"]))
                    print("可见性问题：每条问题一行（未标出问题）" if not label["observability_issues"] else "可见性问题：每条问题一行")
                    display(pd.DataFrame(label["observability_issues"], columns=["type","affected_area","detail"]).rename(columns={"type":"类型","affected_area":"影响对象或部位","detail":"具体说明"}))

        if CALL_MODEL == model:
            call = json.loads(label["call_json"]) or {}
            if call.get("request_ref"):
                request = read_call(call["request_ref"])
                body = request["payload"]
                display(JSON({k:v for k,v in body.items() if k != "messages"}))
                for message in body["messages"]:
                    print("消息角色：", message["role"])
                    content = message.get("content", [])
                    if isinstance(content, str):
                        display(Markdown(content))
                    else:
                        for part in content:
                            if part.get("type") == "text":
                                display(Markdown(part["text"]))
                            elif part.get("type") == "image_url":
                                url = part["image_url"]["url"]
                                display(Image(data=base64.b64decode(url.split(",",1)[1]), width=700) if url.startswith("data:") else Markdown(url))
            else:
                print("没有可读取的原生请求引用。")
